# Double Machine Learning : Efecto del tipo de administración sobre el descuento promedio en licitaciones públicas.

## Introducción

En este caso de estudio se analiza el efecto causal del tipo de administración contratante (Local o Central) sobre el descuento promedio obtenido en la adjudicación de los contratos públicos, una vez controladas las características observables de cada procedimiento de contratación. 

Este proyecto se desarrolló a partir del trabajo realizado durante mis prácticas externas. La entidad donde se llevó a cabo el proyecto ha autorizado la publicación de este repositorio, preservando la confidencialidad de la información utilizada. Por este motivo, los datos originales no se incluyen ni se comparten públicamente.

En este repositorio se documenta la metodología seguida, incluyendo las etapas de preparación, análisis y modelado de los datos, de forma que el proceso pueda reproducirse y adaptarse a otros conjuntos de datos con características similares. Asimismo, se muestran únicamente los resultados del análisis que no contienen información confidencial ni permiten identificar a la entidad o a las personas involucradas.

La base de datos con la que trabajamos dispone inicialmente de 517.149 filas, siendo estas licitaciones, y 37 variables, que representan las características de la licitación. Algunas de estas características son: sistema de contratación, tipo de procedimiento, tipo de administración, CPV imputado, etc.

Para ello, se emplea el algoritmo **Double Machine Learning (DML)** para el modelo parcialmente lineal.


La pregunta causal que se pretende responder es:

> **¿Cuál es el efecto de que una licitación sea convocada por una Administración Central, en lugar de una Administración Local, sobre el descuento promedio obtenido en la adjudicación, una vez controladas las características observables del contrato?**

## 1. Importación de librerías

En esta sección se importan las librerías necesarias para la manipulación y el preprocesamiento de los datos. Asimismo, se fija una semilla aleatoria para garantizar que el análisis pueda reproducirse en futuras ejecuciones.

In [ ]:
# 1. IMPORTACIÓN DE LIBRERÍAS

# Manipulación de datos
import numpy as np
import pandas as pd

# Reproducibilidad (semilla)
RANDOM_STATE = 123
np.random.seed(RANDOM_STATE)

## 2. Carga de la base de datos

Se carga la base de datos empleada en el análisis y se realiza una primera inspección con el fin de comprobar que la lectura se ha realizado correctamente y conocer la estructura del conjunto de datos.


In [ ]:
# 2. CARGA DE LA BASE DE DATOS

df = pd.read_csv("base_datos_lector.csv) #el lector pondrá aquí el nombre de su fichero 

print(f"Número de observaciones: {df.shape[0]}")
print(f"Número de variables: {df.shape[1]}")


## 3. Exploración incial de los datos

In [ ]:
df.head() #primeras observaciones
df.info() #información general sobre las variables
df.describe(include="all") #resumen estadístico

## 4. Definición del problema causal.

### 4.1. Definición de la variable respuesta

La variable respuesta del estudio es el **descuento promedio obtenido en la adjudicación**, entendido como la diferencia relativa entre el presupuesto base de licitación y el importe finalmente adjudicado.

Esta variable constituye la medida de interés sobre la que se pretende evaluar el efecto causal del nivel administrativo del órgano de contratación. Valores elevados indican que la adjudicación se ha realizado con un mayor descuento respecto al presupuesto inicialmente previsto.

In [ ]:
Y = df["descuento_promedio"].to_numpy()

### 4.2 Definición de la variable tratamiento.


La variable de tratamiento se obtiene a partir de la variable **tipo de administracion**, que identifica el nivel administrativo del órgano de contratación.

Se consideran como grupo de tratamiento las licitaciones convocadas por administraciones centrales, mientras que el grupo de control está formado por aquellas promovidas por administraciones locales.

Con el fin de realizar una comparación homogénea, se excluyen del análisis las observaciones correspondientes a otros niveles administrativos.

In [ ]:
# Categorías correspondientes a la Administración Central

administracion_central = [
    "autoridad estatal",
    "organismo de derecho público bajo el control de una autoridad estatal",
    "empresa pública bajo el control de una autoridad estatal"
]

# Categorías correspondientes a la Administración Local

administracion_local = [
    "autoridad local",
    "organismo de derecho público bajo el control de una autoridad local",
    "empresa pública bajo el control de una autoridad local"
]

In [ ]:
# Conservamos únicamente las administraciones centrales y locales

df = (
    df[
        df["tipo_administracion"].isin(
            administracion_central + administracion_local
        )
    ]
    .copy()
    .reset_index(drop=True)
)

In [ ]:
# Variable de tratamiento
# 1 = Administración Central
# 0 = Administración Local

df["tratamiento"] = (
    df["tipo_administracion"]
      .isin(administracion_central)
      .astype(int)
)

D = df["tratamiento"].to_numpy()

In [ ]:
# Distribución del tratamiento

df["tratamiento"].value_counts()

### 4.3. Selección de covariables

Con el objetivo de controlar las diferencias observables entre las licitaciones, se selecciona un conjunto de covariables que describen las características del procedimiento de contratación, el objeto del contrato y sus principales características económicas y administrativas. Las covariables son previas a la asignación del tratamiento.

Estas variables constituyen el conjunto de información utilizado por los algoritmos de aprendizaje automático para estimar las funciones auxiliares del modelo de Double Machine Learning.

In [5]:
covariables = [
    "presupuesto_base_sin_impuestos",
    "lote_presupuesto_base_sin_impuestos",
    "valor_estimado_imputado",
    "tipo_contrato",
    "cpv_final_imputado",
    "sistema_contratacion",
    "tipo_procedimiento",
    "es_loteado",
    "lote",
    "lote_pyme",
    "anio",
    "mes",
    "com_aut_licitacion",
    "com_aut_adjudicador"
]

**Codificación de las variables categóricas**

Con el objetivo de utilizar algoritmos de aprendizaje automático que requieren variables numéricas como entrada, las covariables de naturaleza categórica se transforman mediante *frequency encoding*.

Esta técnica sustituye cada categoría por su frecuencia relativa de aparición en el conjunto de datos.

In [ ]:
# Identificación de variables categóricas y numéricas

cat_cols = [
    c for c in covariables
    if df[c].dtype == "object"
]

num_cols = [
    c for c in covariables
    if c not in cat_cols
]

In [ ]:
# Frequency Encoding

for col in cat_cols:

    frecuencias = df[col].value_counts(normalize=True)

    df[col + "_freq"] = df[col].map(frecuencias)

In [ ]:
## Reagrupación de matrices de covariables, de respuesta y de interés
X = df[
    num_cols +
    [c + "_freq" for c in cat_cols]
].to_numpy()

Y = df["descuento_promedio"].to_numpy()

D = df["tratamiento"].to_numpy()

## 5. Especificación de los modelos de aprendizaje automático

El modelo parcialmente lineal de Double Machine Learning requiere estimar dos funciones auxiliares: la esperanza condicional de la variable respuesta y la probabilidad condicional de recibir el tratamiento.

Con el fin de analizar la robustez de los resultados frente a la elección del algoritmo de aprendizaje automático, se consideran diferentes modelos basados en árboles de decisión. En todos los casos se utilizan los mismos hiperparámetros durante la estimación para facilitar la comparación de los resultados.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

In [ ]:
# MODELOS PARA LA VARIABLE RESPUESTA
outcome_models = {

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        random_state=123
    ),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=123
    )

}

In [ ]:
# MODELOS PARA EL TRATAMIENTO

treatment_models = {

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=123
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=123
    )

}

## 6. Estimación mediante Double Machine Learning

Se estima un modelo parcialmente lineal utilizando la librería `DoubleML`.

Con el objetivo de evaluar la sensibilidad de los resultados frente a la elección del algoritmo de aprendizaje automático, se consideran distintas combinaciones entre los modelos utilizados para estimar la variable respuesta y la variable de tratamiento.

En todos los casos se emplea un procedimiento de *5-fold cross-fitting*.

In [ ]:
from doubleml import DoubleMLPLR

resultados = []

for nombre_y, modelo_y in outcome_models.items():

    for nombre_d, modelo_d in treatment_models.items():

        modelo = DoubleMLPLR(
            dml_data,
            ml_l=modelo_y,
            ml_m=modelo_d,
            n_folds=5
        )

        modelo.fit()

        resultados.append({

            "Modelo Y": nombre_y,
            "Modelo T": nombre_d,
            "ATE": modelo.coef[0],
            "SE": modelo.se[0],
            "IC inferior": modelo.confint().iloc[0,0],
            "IC superior": modelo.confint().iloc[0,1]

        })

In [ ]:
resultados = pd.DataFrame(resultados)

resultados

In [ ]:
resultados["ATE (IC 95%)"] = (
    resultados["ATE"].round(6).astype(str)
    + " ["
    + resultados["IC inferior"].round(6).astype(str)
    + ", "
    + resultados["IC superior"].round(6).astype(str)
    + "]"
)

tabla_final = resultados[
    ["Modelo Y","Modelo T","ATE (IC 95%)"]
]

tabla_final

![Resultado](dml_tabla.png)

El efecto medio del tratamiento (ATE) es negativo en las cuatro combinaciones de algoritmos analizadas, con valores comprendidos entre -0,016 y -0,020. Además, los intervalos de confianza al 95 % no incluyen el valor cero, lo que indica que el efecto es estadísticamente significativo. Estos resultados sugieren que, una vez controladas las características observables del contrato, las licitaciones convocadas por la Administración Central obtienen, en promedio, un descuento en la adjudicación entre 1,6 y 2,0 puntos porcentuales inferior al de las licitaciones convocadas por la Administración Local.

En consecuencia, existe evidencia de que las Administraciones Locales consiguen, en promedio, mayores descuentos en la adjudicación que las Administraciones Centrales.

En este proyecto se han analizado dos algoritmos a través de cuatro combinaciones de configuración. Si bien estas permiten realizar una comparación representativa, existen muchas otras combinaciones y enfoques que podrían evaluarse. Su estudio se propone como una posible línea de trabajo para futuros proyectos.
